# Regresja liniowa

### Przygotowanie środowiska programistycznego

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pylab as py

#Increase plots font size
params = {'legend.fontsize': 'xx-large',
              'figure.figsize': (10, 7),
              'axes.labelsize': 'xx-large',
              'axes.titlesize':'xx-large',
              'xtick.labelsize':'xx-large',
              'ytick.labelsize':'xx-large'}
plt.rcParams.update(params)

# Zapoznanie się z regresją liniową
* W ramach tego ćwiczenia będziemy chcieli opisać zbiór danych modelem liniowym.
* Zbiór danych stworzymy sami w sposób sztuczny, ale w typowych problemach zebranie i obróbka danych stanowi znaczącą część pracy.
* Nasz liniowy model ma postać: $y = \theta_0 + \theta_1 x$
* Dane wytworzymy dla konkretnych $\theta_0$ i $\theta_1$, a następnie zaimplementujemy regresję liniową, aby znaleźć jak najlepsze oszacowanie dla tych parametrów.
* `(X,Y)` to ciąg uczący. *Co to ciąg uczący?*

## Produkcja danych

Dane wytorzymy według liniowej zależnośći

$$
{\Huge
y = \theta_0 + \theta_1 \cdot x
}
$$

Ustalamy parametry dla symulacji na $\theta_0 = 1$ i $\theta_1 = 3$. Dla wygody włóżmy oba parametry do wektora (np.array):

$$
{\Huge
\vec{\theta} = (\theta_{0}, \theta_{1})
}
$$

In [3]:
theta0 = 1
theta1 = 3
theta = np.array([theta0, theta1])

Użyjemy **nPoints** równoodległych punktów $x$ i dla nich wygenerujmy punkty wg. założonego modelu. 

Dla wygody dane załadujemy do obiektu DataFrame. By to zrobić musimy zmienić wektor wierszowy na kolumnowy. 

In [4]:
nPoints = 100
x = np.linspace(0, 10, nPoints)

df = pd.DataFrame(data=x.T, columns = ["x"])
df["y"] = theta[0] + df["x"]*theta[1]

**Proszę:**

* do danych "czystych" dodać kolumnę z danymi z szumem Gaussowskim: $$ y_{noise} = y + Rand(N(0,1)) $$
* Jako ciekawostka wyjaśnienie różnicy między dwoma sposobami generowania liczb losowych [tutaj](https://builtin.com/data-science/numpy-random-seed#:~:text=default_rng()%20creates%20isolated%20generators,random%20numbers%2C%20ensuring%20reproducible%20results).

In [5]:
#BEGIN_SOLUTION
df["y_noise"] = df["y"] + np.random.default_rng().normal(size=nPoints)
#END_SOLUTION
print(df)

           x          y    y_noise
0    0.00000   1.000000   1.657387
1    0.10101   1.303030   0.801711
2    0.20202   1.606061   2.073642
3    0.30303   1.909091   2.235912
4    0.40404   2.212121   2.711200
..       ...        ...        ...
95   9.59596  29.787879  29.487117
96   9.69697  30.090909  29.641726
97   9.79798  30.393939  28.210384
98   9.89899  30.696970  30.434637
99  10.00000  31.000000  30.098269

[100 rows x 3 columns]


Obejrzyjmy te dane. 

**Proszę narysować:**
* na jednym rysunku: `y vs x` oraz `y_noise vs x`
* na drugim rysunku: histogram `y - y_noise`   

In [94]:
#BEGIN_SOLUTION
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=df["x"], y=df["y"], mode='lines', name='True function'))
fig.add_trace(go.Scatter(x=df["x"], y=df["y_noise"], mode='markers', name='Noisy data'))
fig.update_layout(title='True function and noisy data',
                  xaxis_title='x',
                  yaxis_title='y')
fig.show()

fig = go.Figure()
fig.add_trace(go.Histogram(x=df["y_noise"], name='Residuals', nbinsx=30))
fig.update_layout(title='Distribution of residuals',
                  xaxis_title='Residuals (y - y_pred)',
                  yaxis_title='Number of events')
fig.show()
#END_SOLUTION


## Algorytm gradientowy stochastyczny 

**Proszę** napisać funkcję ```iterative_stochastic_gradient(x,y, theta, alpha, nIter)``` która:
* na wejściu przyjmuje ciąg uczący w postaci wektorów x i y, wartości początkowe $(\theta_{0}, \theta_{1})$, parametr szybkości zbiegania $\alpha$ oraz liczbę iteracji `nIter`
* implementuje wzór na parametry optymalne na podstawie [algorytmu najmniejszych kwadratów](https://kampus-student2.ckc.uw.edu.pl/mod/url/view.php?id=247878). 
* zwraca estymowane parametry $(\theta_{0}^{est}, \theta_{1}^{est})$ dla wszystkich iteracji, czyli tablicę o kształcie
 $(nIter+1, 2)$. Początkową wartość $(\theta_{0}^{est}, \theta_{1}^{est})$ także należy dołączyć, stąd `nIter+1` elementów
* funkcja powinna być przetestowana na czystych danych, czyli parze $(x,y)$ z poczatkową wartością $(\theta_{0}, \theta_{1})$ **równą** nominalnej
* funkcja powinna być przetestowana na czystych danych, czyli parze $(x,y)$ z poczatkową wartością $(\theta_{0}, \theta_{1})$ **różną** od nominalnej

In [11]:
%%time
def iterative_stochastic_gradient(x, y, init_theta, alpha, nIter):
    
    theta = init_theta
    theta_est = np.copy(theta)
    theta_est_hist = [theta]

    
    indices = np.random.randint(low=0, high=len(x)-1, size=(nIter))
    for iteration in range(nIter):
        #BEGIN_SOLUTION
        x_sample = x[indices[iteration]]
        x_sample = np.column_stack((np.ones(1), x_sample)) # Zastanów się dlaczego dodajemy kolumnę jedynek
        y_sample = y[indices[iteration]]
        theta = theta - alpha*(np.dot(theta, x_sample.T) - y_sample)*x_sample
        #END_SOLUTION
        theta_est_hist.append(np.copy(theta))
    return np.vstack(theta_est_hist) 
        
          
theta_est_hist = iterative_stochastic_gradient(df["x"], df["y"], theta, 0.01,1000)    
print("Wartości estymowane z użyciem nominalnej wartości początkowej parametrów theta:",theta_est_hist[-1])

theta_est_hist = iterative_stochastic_gradient(df["x"], df["y"], theta+3, 0.01, 1000)    
print("Wartości estymowane z użyciem różnej od nominalnej wartości początkowej parametrów theta:",theta_est_hist[-1])

Wartości estymowane z użyciem nominalnej wartości początkowej parametrów theta: [1. 3.]
Wartości estymowane z użyciem różnej od nominalnej wartości początkowej parametrów theta: [1.2260086  2.96297105]
CPU times: user 16.4 ms, sys: 0 ns, total: 16.4 ms
Wall time: 16.4 ms


**Proszę** narysować następujące rysunki:
* dane, oraz krzywe regresji dla 10 iteracji na jednym rysunku
* wartości parametrów $\theta_{0}$ i $\theta_{1}$ w funkcji numeru iteracji dla $\alpha$ = 0.1, 0.05 oraz 0.01 oraz 10 iteracji
* wartości parametrów $\theta_{0}$ i $\theta_{1}$ w funkcji numeru iteracji dla $\alpha$ = 0.01 dla 100 iteracji

W każdym przypadku jako wartości początkowe proszę przyjąć ($\theta_{0}$, $\theta_{1}$) + 1

In [95]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

#BEGIN_SOLUTION
theta_est = iterative_stochastic_gradient(df["x"], df["y_noise"], theta+1, 0.01, 10)

x = df["x"]
x_aug = np.column_stack((np.ones(x.shape[0]), x))

y_fit = theta_est.dot(x_aug.T).T

theta_alpha_01 = iterative_stochastic_gradient(df["x"], df["y_noise"], theta+1, alpha=0.1, nIter=10)
theta_alpha_005 = iterative_stochastic_gradient(df["x"], df["y_noise"], theta+1, alpha=0.05, nIter=10)
theta_alpha_001 = iterative_stochastic_gradient(df["x"], df["y_noise"], theta+1, alpha=0.01, nIter=10)
theta_alpha = iterative_stochastic_gradient(df["x"], df["y_noise"], theta+1, alpha=0.01, nIter=100)

# --- tworzymy siatkę 2x2 ---
fig = make_subplots(rows=2, cols=2, subplot_titles=[
    "Dane i dopasowania", 
    "theta_0 vs iteracje",
    "theta_1 vs iteracje",
    "theta_0, theta_1 dla α=0.01"
])

# --- subplot (1,1) dane i linie dopasowane ---
fig.add_trace(go.Scatter(x=df["x"], y=df["y_noise"], mode="markers", name="data", marker=dict(color="red")), row=1, col=1)
fig.add_trace(go.Scatter(x=df["x"], y=y_fit[:,0], mode="lines", line=dict(color="green", width=3), name="initial params"), row=1, col=1)
for i in range(1, y_fit.shape[1]-1):
    fig.add_trace(go.Scatter(x=df["x"], y=y_fit[:,i], mode="lines", line=dict(color="blue", width=1), showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=df["x"], y=y_fit[:,-1], mode="lines", line=dict(color="yellow", width=3), name="final params"), row=1, col=1)

# --- subplot (1,2) theta_0 ---
fig.add_trace(go.Scatter(y=theta_alpha_01[:,0], mode="lines", line=dict(width=3), name="α=0.1"), row=1, col=2)
fig.add_trace(go.Scatter(y=theta_alpha_005[:,0], mode="lines", line=dict(width=3), name="α=0.05"), row=1, col=2)
fig.add_trace(go.Scatter(y=theta_alpha_001[:,0], mode="lines", line=dict(width=3), name="α=0.01"), row=1, col=2)

# --- subplot (2,1) theta_1 ---
fig.add_trace(go.Scatter(y=theta_alpha_01[:,1], mode="lines", line=dict(width=3), name="α=0.1"), row=2, col=1)
fig.add_trace(go.Scatter(y=theta_alpha_005[:,1], mode="lines", line=dict(width=3), name="α=0.05"), row=2, col=1)
fig.add_trace(go.Scatter(y=theta_alpha_001[:,1], mode="lines", line=dict(width=3), name="α=0.01"), row=2, col=1)

# --- subplot (2,2) theta_0 i theta_1 dla α=0.01 ---
fig.add_trace(go.Scatter(y=theta_alpha[:,0], mode="lines", line=dict(color="red", width=3), name="theta_0, α=0.01"), row=2, col=2)
fig.add_trace(go.Scatter(y=theta_alpha[:,1], mode="lines", line=dict(color="blue", width=3), name="theta_1, α=0.01"), row=2, col=2)
fig.add_trace(go.Scatter(y=[theta[0]]*len(theta_alpha), mode="lines", line=dict(color="red", dash="dash", width=2), showlegend=False), row=2, col=2)
fig.add_trace(go.Scatter(y=[theta[1]]*len(theta_alpha), mode="lines", line=dict(color="blue", dash="dash", width=2), showlegend=False), row=2, col=2)

# --- layout ---
fig.update_layout(height=800, width=900, showlegend=False)

fig.show()
#END_SOLUTION

## Algorytm gradientowy zbiorczy

**Proszę** napisać funkcję ```iterative_batch_gradient(x,y, theta, alpha, nIter)``` która:
* na wejściu przyjmuje ciąg uczący w postaci wektorów ```x``` i ```y```, wartości początkowe $(\theta_{0}, \theta_{1})$, 
  parametr szybkości zbiegania $\alpha$ oraz liczbę iteracji ```nIter```
* implementuje wzór na parametry optymalne na podstawie [algorytmu najmniejszych kwadratów](https://kampus-student2.ckc.uw.edu.pl/mod/url/view.php?id=247878). 
* zwraca estymowane parametry $(\theta_{0}^{est}, \theta_{1}^{est})$ dla wszystkich iteracji, czyli tablicę o kształcie
 $(nIter+1, 2)$. Początkową wartość $(\theta_{0}^{est}, \theta_{1}^{est})$ także należy dołączyc, stąd nIter+1 elementów
* funkcja powinna być przetestowana na czystych danych, czyli parze $(x,y)$ z poczatkową wartością $(\theta_{0}, \theta_{1})$ **równą** nominalnej
* funkcja powinna być przetestowana na czystych danych, czyli parze $(x,y)$ z poczatkową wartością $(\theta_{0}, \theta_{1})$ **różną** od nominalnej

In [27]:
%%time

def iterative_batch_gradient(x, y, init_theta, alpha, nIter):
    theta = init_theta
    theta_est = np.copy(theta)
    theta_est = np.reshape(theta_est, (-1,2))
    batchSize = len(x)
    for iteration in range(nIter):  
        #BEGIN_SOLUTION 
        x_batch = np.column_stack((np.ones(batchSize), x))
        y_batch = y   
        delta = 2.0*alpha*(np.sum(theta*x_batch, axis=1) - y_batch)
        delta = np.array(delta)
        delta = np.reshape(delta, (batchSize,1))     
        delta = delta*x_batch     
        delta = np.sum(delta, axis=0)/batchSize
        delta = np.reshape(delta, (1,-1)) 
        theta = theta - delta
        #END_SOLUTION
        theta_est = np.append(theta_est, theta, axis=0)
    return theta_est   
        
theta_est = iterative_batch_gradient(df["x"], df["y"], init_theta=theta, alpha=0.01, nIter=1)    
print("Wartości estymowane z użyciem nominalnej wartości początkowej parametrów theta:",theta_est[-1])

theta_est = iterative_batch_gradient(df["x"], df["y"], init_theta=theta+1, alpha=0.01, nIter=500)    
print("Wartości estymowane z użyciem różnej od nominalnej wartości początkowej parametrów theta:",theta_est[-1])

Wartości estymowane z użyciem nominalnej wartości początkowej parametrów theta: [1. 3.]
Wartości estymowane z użyciem różnej od nominalnej wartości początkowej parametrów theta: [1.06901228 2.9896233 ]
CPU times: user 83 ms, sys: 1.08 ms, total: 84 ms
Wall time: 79.4 ms


**Proszę** narysować następujące rysunki:
* dane oraz krzywe regressji dla 10 iteracji na jednym rysunku
* wartości parametrów $\theta_{0}$ i $\theta_{1}$ w funkcji numeru iteracji dla $\alpha$ = 0.1, 0.05 oraz 0.01 oraz 10 iteracji
* wartości parametrów $\theta_{0}$ i $\theta_{1}$ w funkcji numeru iteracji dla $\alpha$ = 0.01 dla 100 iteracji

W każdym przypadku jako wartości początkowe proszę przyjąć ($\theta_{0}$, $\theta_{1}$) + 1

In [96]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

#BEGIN_SOLUTION
theta_est = iterative_batch_gradient(df["x"], df["y_noise"], theta+1, 0.01, 10)

x = df["x"]
x_aug = np.column_stack((np.ones(x.shape[0]), x))

y_fit = theta_est.dot(x_aug.T).T

theta_alpha_01 = iterative_batch_gradient(df["x"], df["y_noise"], theta+1, alpha=0.1, nIter=10)
theta_alpha_005 = iterative_batch_gradient(df["x"], df["y_noise"], theta+1, alpha=0.05, nIter=10)
theta_alpha_001 = iterative_batch_gradient(df["x"], df["y_noise"], theta+1, alpha=0.01, nIter=10)
theta_alpha = iterative_batch_gradient(df["x"], df["y_noise"], theta+1, alpha=0.01, nIter=100)

# --- subploty 2x2 ---
fig = make_subplots(rows=2, cols=2, subplot_titles=[
    "Dane i dopasowania", 
    "theta_0 vs iteracje",
    "theta_1 vs iteracje",
    "theta_0, theta_1 dla alpha=0.01"
])

# --- subplot (1,1) dane i linie dopasowane ---
fig.add_trace(go.Scatter(x=df["x"], y=df["y_noise"], mode="markers",
                         name="data", marker=dict(color="red")), row=1, col=1)

fig.add_trace(go.Scatter(x=df["x"], y=y_fit[:,0], mode="lines",
                         line=dict(color="green", width=3), name="initial params"), row=1, col=1)

for i in range(1, y_fit.shape[1]-1):
    fig.add_trace(go.Scatter(x=df["x"], y=y_fit[:,i], mode="lines",
                             line=dict(color="blue", width=1), showlegend=False), row=1, col=1)

fig.add_trace(go.Scatter(x=df["x"], y=y_fit[:,-1], mode="lines",
                         line=dict(color="yellow", width=3), name="final params"), row=1, col=1)

# --- subplot (1,2) theta_0 ---
fig.add_trace(go.Scatter(y=theta_alpha_01[:,0], mode="lines", line=dict(width=3), name="alpha=0.1"), row=1, col=2)
fig.add_trace(go.Scatter(y=theta_alpha_005[:,0], mode="lines", line=dict(width=3), name="alpha=0.05"), row=1, col=2)
fig.add_trace(go.Scatter(y=theta_alpha_001[:,0], mode="lines", line=dict(width=3), name="alpha=0.01"), row=1, col=2)

# --- subplot (2,1) theta_1 ---
fig.add_trace(go.Scatter(y=theta_alpha_01[:,1], mode="lines", line=dict(width=3), name="alpha=0.1"), row=2, col=1)
fig.add_trace(go.Scatter(y=theta_alpha_005[:,1], mode="lines", line=dict(width=3), name="alpha=0.05"), row=2, col=1)
fig.add_trace(go.Scatter(y=theta_alpha_001[:,1], mode="lines", line=dict(width=3), name="alpha=0.01"), row=2, col=1)

# --- subplot (2,2) theta_0 i theta_1 dla alpha=0.01 ---
fig.add_trace(go.Scatter(y=theta_alpha[:,0], mode="lines", line=dict(color="red", width=3),
                         name="theta_0, alpha=0.01"), row=2, col=2)
fig.add_trace(go.Scatter(y=theta_alpha[:,1], mode="lines", line=dict(color="blue", width=3),
                         name="theta_1, alpha=0.01"), row=2, col=2)

fig.add_trace(go.Scatter(y=[theta[0]]*len(theta_alpha), mode="lines",
                         line=dict(color="red", dash="dash", width=2), showlegend=False), row=2, col=2)
fig.add_trace(go.Scatter(y=[theta[1]]*len(theta_alpha), mode="lines",
                         line=dict(color="blue", dash="dash", width=2), showlegend=False), row=2, col=2)

# --- layout ---
fig.update_layout(
    height=800, width=900, 
    title_text="Regresja liniowa - batch gradient",
    showlegend=False,
    xaxis_title="x",
    yaxis_title="y"
)

fig.show()
#END_SOLUTION

# --- wypisywanie wyników ---
print("Finalna wartość parametrów dla alpha=0.1, nIter = 10:\t", theta_alpha_01[-1])
print("Finalna wartość parametrów dla alpha=0.05, nIter = 10:\t", theta_alpha_005[-1])
print("Finalna wartość parametrów dla alpha=0.01, nIter = 10:\t", theta_alpha_001[-1])
print("Finalna wartość parametrów dla alpha=0.01, nIter = 100:\t", theta_alpha[-1])
print("Oryginalna wartość parametrów:\t\t\t", theta)


Finalna wartość parametrów dla alpha=0.1, nIter = 10:	 [ 8055941.04471281 53577614.82999447]
Finalna wartość parametrów dla alpha=0.05, nIter = 10:	 [1208.86051861 8031.65184711]
Finalna wartość parametrów dla alpha=0.01, nIter = 10:	 [1.79042139 2.86574338]
Finalna wartość parametrów dla alpha=0.01, nIter = 100:	 [1.52035469 2.90633947]
Oryginalna wartość parametrów:			 [1 3]


# Zadanie domowe

**Proszę:**


1. **Stwórz dane syntetyczne:**  
   - `X` w zakresie od 0 do 10, np. `X = np.linspace(0, 10, 100)`  
   - `y = 3 + 6 * X`, gdzie `theta_true = [3, 6]`  

2. **Ustaw parametry regresji liniowej:**  
   - Początkowe `theta_init = theta_true + 1`  
   - Współczynnik uczenia `alpha = 0.01`  
   - Liczba iteracji `nIter = 1000`  
   - Liczba poziomów szumu `nNoise = 50`  

3. **Stopniowe zaszumianie danych:**  
   - W pętli dla `i` od 0 do `nNoise-1` utwórz:
   
     ```python
     y_noise = y + i * np.random.randn(len(X))
     ```
   - Dla każdego `y_noise` dopasuj regresję liniową metodą batch gradient descent (`iterative_batch_gradient`)  
   - Zapisz finalne wartości theta w każdej iteracji

4. **Wizualizacja wyników:**  
   - Narysuj dopasowane proste dla wszystkich poziomów szumu  
   - **Uwaga:** Wybierz jeden kolor linii i uzależnij intensywność koloru od poziomu zaszumienia sygnału. Moża to zrobić na dużo sposobów.
   - Narysuj osobny wykres zmian `theta_0` i `theta_1` w funkcji poziomu szumu

---

## Dodatkowe pytanie

- Spróbuj zmienić zakres `X`, np. na:
  ```python
  X = np.linspace(0, 100, 1000)
  ```
  Zastanów się: co trzeba zmienić w kodzie, aby gradient descent nadal działał stabilnie?

In [92]:
import numpy as np
import plotly.graph_objects as go
import matplotlib.cm as cm  # do generowania kolorów z colormap
import matplotlib.colors as mcolors

#BEGIN_SOLUTION
# --- Generowanie danych ---
theta_true = np.array([3, 6])
X = np.linspace(0, 10, 100)
y = theta_true[0] + theta_true[1]*X
# print(y)
theta_init = theta_true + 1
alpha = 0.01
nIter = 1000
nNoise = 50

# paleta kolorów matplotlib
norm = mcolors.Normalize(vmin=0, vmax=nNoise)  # normalizacja poziomu szumu
cmap = cm.get_cmap('viridis')  # można też 'plasma', 'inferno', 'cool'

# --- Stopniowe zaszumianie i dopasowanie ---
theta_all = []

fig = go.Figure()

for i in range(nNoise-1):
    y_noise = y + i * np.random.randn(len(X))
    theta_hist = iterative_batch_gradient(X.copy(), y_noise.copy(), theta_init.copy(), alpha, nIter)
    # print("AA", theta_hist)
    theta_all.append(theta_hist[-1])  # zapisujemy finalne theta
    # dodajemy linię dopasowaną dla tego poziomu szumu
    y_fit = theta_hist[-1,0] + theta_hist[-1,1]*X

    # kolor zależny od poziomu szumu
    color_rgb = mcolors.to_hex(cmap(norm(i)))
    fig.add_trace(go.Scatter(x=X, y=y_fit, mode='lines', 
                             name=f'Noise level {i}', 
                             line=dict(width=2, color=color_rgb)))
fig.show()

# --- Konwersja wyników do tablicy ---
theta_all = np.array(theta_all)

# --- Wykres zmian theta w funkcji poziomu szumu ---
fig2 = go.Figure()
fig2.add_trace(go.Scatter(y=theta_all[:,0], mode='lines+markers', name='theta_0'))
fig2.add_trace(go.Scatter(y=theta_all[:,1], mode='lines+markers', name='theta_1'))

# --- Layouty ---
fig.update_layout(title="Dopasowane proste przy rosnącym szumie",
                  xaxis_title="X", yaxis_title="y")

fig2.update_layout(title="Zmienność parametrów theta w funkcji poziomu szumu",
                   xaxis_title="Poziom szumu", yaxis_title="theta")

fig2.show()
#END_SOLUTION


/tmp/ipykernel_22084/4051391825.py:17: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.

